# DIOR Faster R-CNN: Colab setup and training

Use a Colab GPU runtime. Run setup cells in order. The smoke test trains on small subsets for one epoch; it verifies data loading, forward/backward passes, validation loss, and checkpoint saving. It is not a final experiment.

Dataset ZIPs are downloaded from the public Drive folder into the temporary Colab VM. No manual upload of extracted folders is needed. Checkpoints are saved to your mounted Drive.

In [ ]:
# Clone the project code from GitHub. This repo is fetched by URL, so GitHub OAuth is not required for a public repo.
!test -d /content/ecs174-dior/.git || git clone https://github.com/viveknarayana/ecs174-dior.git /content/ecs174-dior
%cd /content/ecs174-dior
import torch
print('GPU:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'NO GPU')

In [ ]:
# Download the four public dataset ZIPs (~7 GB) and extract them in Colab.
!pip -q install gdown
!mkdir -p /content/dior_archives /content/DIOR
!gdown --folder 'https://drive.google.com/drive/folders/1UdlgHk49iu6WpcJ5467iT-UqNPpx__CC' -O /content/dior_archives

from pathlib import Path
from zipfile import ZipFile
archive_dir = Path('/content/dior_archives')
data_root = Path('/content/DIOR')
for archive in sorted(archive_dir.glob('*.zip')):
    print('Extracting', archive.name)
    with ZipFile(archive) as zf:
        zf.extractall(data_root)

assert (data_root / 'JPEGImages-trainval').is_dir(), 'JPEGImages-trainval folder not found after extraction'
assert (data_root / 'JPEGImages-test').is_dir(), 'JPEGImages-test folder not found after extraction'
assert (data_root / 'Annotations' / 'Horizontal Bounding Boxes').is_dir(), 'Annotations folder not found after extraction'
assert (data_root / 'ImageSets' / 'Main' / 'train.txt').is_file() or (data_root / 'Main' / 'train.txt').is_file(), 'train.txt split file not found after extraction'
print('DIOR folder layout looks ready:', data_root)

In [ ]:
# Mount Drive so checkpoints survive a Colab runtime disconnect.
from google.colab import drive
drive.mount('/content/drive')
OUTPUT_ROOT = '/content/drive/MyDrive/DIOR_checkpoints'
print('Checkpoint destination:', OUTPUT_ROOT)

## First: smoke test

Run this once before full training. It uses up to 300 train and 64 validation images for one epoch, with batch size 2. Expect the COCO weights to download on the first run. Check that it prints epoch losses and writes `checkpoint_epoch_01.pth` and `best.pth` under `MyDrive/DIOR_checkpoints/resnet50/`.

In [ ]:
!python train.py --data-root /content/DIOR --backbone resnet50 --output-root {OUTPUT_ROOT} --tiny-test --epochs 1 --batch-size 2

## Full runs

After the smoke test succeeds, run one full model cell at a time. Each trains up to 10 epochs on the training split and logs validation loss. Checkpoints are written after each completed epoch.

In [ ]:
# ResNet-50 full run
!python train.py --data-root /content/DIOR --backbone resnet50 --output-root {OUTPUT_ROOT} --epochs 10 --batch-size 2

In [ ]:
# MobileNet full run
!python train.py --data-root /content/DIOR --backbone mobilenet --output-root {OUTPUT_ROOT} --epochs 10 --batch-size 2

## Resume an interrupted run

After reconnecting to a Colab GPU runtime, rerun the repo, data, and Drive setup cells, then use the matching backbone and latest completed epoch checkpoint. For example:

In [ ]:
# Example: resume ResNet-50 from the epoch 3 checkpoint
!python train.py --data-root /content/DIOR --backbone resnet50 --output-root {OUTPUT_ROOT} --epochs 10 --batch-size 2 --resume /content/drive/MyDrive/DIOR_checkpoints/resnet50/checkpoint_epoch_03.pth